# Machine health practice starter
Synthetic learning data. Read the manual first. Run cells in order; training is optional. No live app or database is accessed.

In [ ]:
# Cell 1: Choose the extracted dataset folder.
from pathlib import Path
import json
import pandas as pd
import numpy as np
BASE = Path.cwd()
assert (BASE / 'telemetry_raw.csv').exists(), 'Open the notebook in the extracted data folder'
OUT = BASE / 'outputs'
OUT.mkdir(exist_ok=True)
print('Data folder:', BASE)


In [ ]:
# Cell 2: Preserve raw text and audit before cleaning.
raw = pd.read_csv(BASE / 'telemetry_raw.csv', dtype=str, keep_default_na=False)
registry = pd.read_csv(BASE / 'machine_registry.csv')
crosswalk = pd.read_csv(BASE / 'machine_id_crosswalk.csv', dtype=str)
audit = {'rows': len(raw), 'unique_packets': int(raw.packet_id.nunique()),
         'repeated_packet_rows': int(raw.packet_id.duplicated().sum()),
         'missing_cells': raw.eq('').sum().to_dict(),
         'temperature_units': raw.temperature_unit.value_counts().to_dict(),
         'pressure_units': raw.pressure_unit.value_counts().to_dict(),
         'quality_flags': raw.quality_flag.value_counts().to_dict()}
(OUT / 'raw_audit.json').write_text(json.dumps(audit, indent=2))
display(raw.head(10))
print(json.dumps(audit, indent=2))


In [ ]:
# Cell 3: Normalize identities, timestamps, numbers and units.
df = raw.copy()
mapping = dict(zip(crosswalk.raw_machine_id, crosswalk.canonical_machine_id))
df['machine_id'] = df.machine_id.replace(mapping)
df['observed_at_utc'] = pd.to_datetime(df.observed_at, format='mixed', utc=True, errors='coerce')
numeric = ['temperature','vibration','spindle_rpm','motor_current_a',
           'motor_load_pct','pressure','coolant_flow_l_min',
           'ambient_temperature_c','hours_since_service']
for c in numeric:
    text = df[c].str.replace(',', '.', regex=False) if c == 'vibration' else df[c]
    df[c] = pd.to_numeric(text, errors='coerce').replace(-9999, np.nan)
df['temperature_c'] = df.temperature.where(df.temperature_unit == 'C', (df.temperature - 32)*5/9)
df.loc[~df.temperature_unit.isin(['C','F']), 'temperature_c'] = np.nan
df['pressure_bar'] = df.pressure.where(df.pressure_unit == 'bar', df.pressure/100)
df.loc[~df.pressure_unit.isin(['bar','kPa']), 'pressure_bar'] = np.nan
# Wide example plausibility bounds: NOT machine operating/safety limits.
bounds = {'temperature_c':(-40,180), 'vibration':(0,100),
          'spindle_rpm':(0,100000), 'motor_current_a':(0,1000),
          'motor_load_pct':(0,150), 'pressure_bar':(0,100),
          'coolant_flow_l_min':(0,500), 'ambient_temperature_c':(-40,80),
          'hours_since_service':(0,1000000)}
invalid_counts = {}
for c,(lo,hi) in bounds.items():
    invalid = df[c].notna() & ~df[c].between(lo,hi)
    invalid_counts[c] = int(invalid.sum())
    df.loc[invalid,c] = np.nan
features_numeric = list(bounds)


In [ ]:
# Cell 4: Resolve retransmissions conservatively.
compare = ['packet_id','machine_id','observed_at_utc','machine_state',
           'quality_flag'] + features_numeric
normalized = df[compare].drop_duplicates()
conflict_ids = normalized.loc[normalized.packet_id.duplicated(False), 'packet_id'].unique()
clean = normalized.loc[~normalized.packet_id.isin(conflict_ids)].copy()
invalid_time = clean.observed_at_utc.isna()
unknown_machine = ~clean.machine_id.isin(registry.machine_id)
bad_quality = clean.quality_flag.eq('BAD')
report = {'raw_rows':len(raw), 'normalized_unique_rows':len(normalized),
          'conflicting_packet_ids_excluded':len(conflict_ids),
          'invalid_times':int(invalid_time.sum()),
          'unknown_machines':int(unknown_machine.sum()),
          'bad_quality_rows':int(bad_quality.sum()),
          'invalid_numeric_cells':invalid_counts}
clean = clean.loc[~(invalid_time | unknown_machine | bad_quality)].copy()
assert clean.packet_id.is_unique
clean = clean.sort_values(['machine_id','observed_at_utc'])
clean.to_csv(OUT / 'cleaned_features.csv', index=False)
report['clean_rows'] = len(clean)
(OUT / 'cleaning_report.json').write_text(json.dumps(report, indent=2))
print(report)


In [ ]:
# Cell 5: Join labels one-to-one. Future columns stay outside X.
labels = pd.read_csv(BASE / 'training_labels.csv')
joined = clean.merge(labels[['packet_id','machine_id','health_priority']],
                     on='packet_id', how='left', validate='one_to_one', suffixes=('','_label'))
assert joined.health_priority.notna().all()
assert joined.machine_id.eq(joined.machine_id_label).all()
joined = joined.merge(registry[['machine_id','machine_type','plant']],
                       on='machine_id', validate='many_to_one')
eligible = joined.loc[joined.health_priority.isin(['Normal','Suspect','Critical'])
                     & ~joined.machine_state.eq('DOWN')].copy()


In [ ]:
# Cell 6: Chronological splits with a full 120-minute horizon.
plan = json.loads((BASE / 'split_plan.json').read_text())
splits = {}
for name in ['train','validation','test']:
    start = pd.Timestamp(plan[name]['start_inclusive'])
    end = pd.Timestamp(plan[name]['end_exclusive'])
    mask = (eligible.observed_at_utc >= start) & (eligible.observed_at_utc + pd.Timedelta(minutes=120) < end)
    splits[name] = eligible.loc[mask].sort_values('observed_at_utc').copy()
    assert splits[name].packet_id.is_unique
    assert set(splits[name].health_priority) == {'Normal','Suspect','Critical'}
    splits[name].to_csv(OUT / f'{name}.csv', index=False)
    print(name, len(splits[name]), splits[name].health_priority.value_counts().to_dict())
for a,b in [('train','validation'),('train','test'),('validation','test')]:
    assert not set(splits[a].packet_id) & set(splits[b].packet_id)


In [ ]:
# Cell 7: Initial training-only EDA. Extend it using the manual.
import matplotlib.pyplot as plt
figdir = OUT / 'figures'
figdir.mkdir(exist_ok=True)
train = splits['train']
missing = train[features_numeric].isna().mean().sort_values(ascending=False)
missing.to_csv(OUT / 'training_missingness.csv')
missing.plot.bar(title='Missing fraction in training readings', ylabel='Fraction')
plt.tight_layout(); plt.savefig(figdir / 'missingness.png'); plt.show()
train.temperature_c.hist(bins=40)
plt.title('Training machine temperature'); plt.xlabel('Temperature C'); plt.ylabel('Observations')
plt.tight_layout(); plt.savefig(figdir / 'temperature_histogram.png'); plt.show()
machine = train.machine_id.iloc[0]
part = train.loc[train.machine_id.eq(machine)].sort_values('observed_at_utc')
plt.plot(part.observed_at_utc, part.vibration)
plt.title(f'{machine} training vibration'); plt.ylabel('mm/s'); plt.xlabel('UTC time')
plt.xticks(rotation=30); plt.tight_layout(); plt.savefig(figdir / 'vibration_trend.png'); plt.show()


## Optional baseline training
Run only after reviewing your audit and cleaning policy. These cells train models locally; no result is claimed in advance.

In [ ]:
# Cell 8: All fitted preprocessing belongs inside the pipeline.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix
categorical = ['machine_type','plant']
inputs = features_numeric + categorical
preprocessing = ColumnTransformer([
    ('numeric', SimpleImputer(strategy='median', add_indicator=True), features_numeric),
    ('category', OneHotEncoder(handle_unknown='ignore'), categorical)])
model = Pipeline([('preprocess', preprocessing), ('model', RandomForestClassifier(
    n_estimators=100, max_depth=12, min_samples_leaf=5,
    class_weight='balanced', random_state=42, n_jobs=2))])
X_train = splits['train'][inputs]
y_train = splits['train'].health_priority
dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
model.fit(X_train, y_train)


In [ ]:
# Cell 9: Compare on validation, not the final test.
classes = ['Normal','Suspect','Critical']
val = splits['validation']
for name,estimator in [('majority',dummy),('forest',model)]:
    prediction = estimator.predict(val[inputs])
    metrics = classification_report(val.health_priority, prediction, labels=classes, output_dict=True, zero_division=0)
    (OUT / f'{name}_validation_metrics.json').write_text(json.dumps(metrics, indent=2))
    print(name, classification_report(val.health_priority, prediction, labels=classes, zero_division=0))
    print(confusion_matrix(val.health_priority, prediction, labels=classes))
    result = val[['packet_id','machine_id','observed_at_utc','health_priority']].copy()
    result['predicted_priority'] = prediction
    result.to_csv(OUT / f'{name}_validation_predictions.csv', index=False)


In [ ]:
# Cell 10: Open the final test only after freezing your decisions.
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    test = splits['test']
    prediction = model.predict(test[inputs])
    metrics = classification_report(test.health_priority, prediction, labels=classes, output_dict=True, zero_division=0)
    (OUT / 'final_test_metrics.json').write_text(json.dumps(metrics, indent=2))
    print(classification_report(test.health_priority, prediction, labels=classes, zero_division=0))
    import joblib
    joblib.dump(model, OUT / 'trusted_local_model.joblib')
    (OUT / 'model_input_schema.json').write_text(json.dumps({'features':inputs,
        'classes':classes, 'units':{'temperature_c':'C','pressure_bar':'bar','vibration':'mm/s'}}, indent=2))
else:
    print('Final test remains locked. Complete validation and freeze decisions first.')
